# March Mania · Does the women’s record signal repeat?

**Milestone 05 · frozen feature replication → conditional drop-one tests → checkpoint**

The completed 2018 study lowered women’s Brier from **0.1536501 to 0.1522387** using four schedule-record features. Adding the quality-win family weakened that benefit. These are historical exploratory scores, not a new Kaggle result.

We will not add another feature family before checking this signal. The four formulas, 16 reference inputs, classifier and regularization stay fixed. The first stage requires only **five new classifier fits**; three completed fits are replayed without training. Four-feature drop-one tests run only if the three additional seasons pass the documented compute gate.

Keep the original repository and both earlier research folders unchanged. No environment reinstall, GPU, raw-data download, cloud API calls or submissions are part of this notebook. Choose **Python (March Mania)**.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, FileLink

KIT = Path.cwd().resolve()
if not (KIT / "run_round05.py").is_file():
    KIT = Path.home() / "march_record_validation"
assert (KIT / "run_round05.py").is_file(), "Open this notebook inside march_record_validation."
sys.path.insert(0, str(KIT))
from run_round05 import run_stage
from record_plots import figures
pio.renderers.default = "plotly_mimetype"
print("Kernel:", sys.executable)
print("Research folder:", KIT)
print("Five new replication fits; at most sixteen gated ablation fits. No rating fits.")

## 1 · Read the completed evidence before spending more compute

Negative `delta_vs_anchor` means improved Brier. Women’s `anchor_record` is selected for replication based on the already-seen 2018 result. The men’s experiment, quality-win additions, and shooting panel remain paused—not declared impossible or permanently exhausted.

In [ ]:
prior = pd.read_csv(KIT / "evidence/metrics.csv")
display(prior[["Gender", "Season", "recipe", "brier", "delta_vs_anchor"]].round(7))
print("Source: the uploaded milestone_04_return.zip; no new fits generated this table.")

## 2 · Reuse completed feature construction

Preparation verifies the environment, source state, raw files, seven women’s base snapshots, six schedule snapshots, and three prior classifiers. It builds only the missing **2019 schedule snapshot with identical formulas**; it does not refit any strength/efficiency model. Prior raw data and models stay read-only.

The unchanged record features are total wins above the fixed reference expectation, nonhome surplus, difficult-win credit, and costly-loss burden. The reference is the season’s 75th-percentile strength—not an official bubble team. The formulas are heuristic and not calibrated probabilities. Seed and regular-season information refer to the post-field-announcement pre-tournament snapshot. They are not pregame backtests of regular-season games.

**Hard ceiling: 300 seconds. Heartbeats: 15 seconds.** These are limits, not runtime estimates.

In [ ]:
run_stage("prepare", max_seconds=300)
RUN = Path(json.loads((KIT / "reports/latest_run.json").read_text())["run_dir"])
print(json.dumps(json.loads((RUN / "prepare.json").read_text()), indent=2))
registry = pd.read_csv(RUN / "feature_registry.csv")
display(registry.loc[registry.family == "record", ["feature", "description", "availability"]])

## 3 · Replicate the complete four-feature family unchanged

| Validation season | Earlier tournament training seasons | New classifiers | Existing classifiers replayed |
|---|---|---:|---:|
| 2016 | 2013–2015 | 2 | 0 |
| 2017 | 2013–2016 | 2 | 0 |
| 2018 · discovery | 2013–2017 | 0 | 2 |
| 2019 | 2013–2018 | 1 | 1 |

Only `anchor` (16 inputs) and `anchor_record` (20 inputs) are compared. Logistic C=0.1, training-only RMS scaling, mirrored observations and unit total physical-game weight remain fixed. No ranking-based feature selection, calibration tuning or ensembling is introduced.

**2016–2019 are already-consumed exploratory history.** The extra seasons are not independent fresh tests; 2018 selected this hypothesis, so it is excluded from the compute gate. Only earlier labels enter each fit. No 2020–2026 targets are used.

**Hard ceiling: 180 seconds.**

In [ ]:
run_stage("replicate", max_seconds=180)
metrics = pd.read_csv(RUN / "replication_metrics.csv")
display(metrics[["Season", "role", "recipe", "games", "train_games", "brier", "delta_vs_anchor", "source"]].round(7))
gate = json.loads((RUN / "gate.json").read_text())
print(json.dumps(gate, indent=2))

## 4 · Conditional feature attribution—not automatic expansion

The next cell spends additional compute only when, across **2016, 2017 and 2019**, all three conditions hold:

* Mean Brier change is **at most −0.0005**.
* At least **two of three seasons** improve.
* The worst season worsens by **no more than +0.003**.

These frozen thresholds allocate compute; they are not statistical significance thresholds. A failed gate produces `SKIPPED_BY_GATE` and zero ablation fits. That is a valid completed milestone: continue through reporting, do not change thresholds to make the gate pass.

A passed gate runs four drop-one versions of the unchanged family across all four seasons: **at most 16 new classifiers**, 19 inputs each. Each version removes exactly one record feature while preserving every reference input. No replacement features are selected. This identifies conditional predictive contribution, not causation. No subset is automatically promoted from these exploratory comparisons.

**Hard ceiling: 240 seconds.**

In [ ]:
run_stage("ablate", max_seconds=240)
print(json.dumps(json.loads((RUN / "ablation_receipt.json").read_text()), indent=2))
ablations = pd.read_csv(RUN / "ablation_metrics.csv")
if ablations.empty:
    print("The gate did not justify ablations. Preserve the result and finish the report.")
else:
    display(ablations[["Season", "removed_feature", "brier", "delta_vs_full", "delta_vs_anchor"]].round(7))
    print("Positive delta_vs_full means removing that feature made the full family worse.")

## 5 · Publish saved evidence and preservation checks

The HTML contains the measured scores, gate result, all available Plotly figures and limitations. The return ZIP contains aggregate metrics/provenance only; raw competition rows, fitted models and per-game predictions remain private.

**Hard ceiling: 120 seconds.** A successful status means the pipeline worked, not that the features improved. Historical scores must not be compared directly with the 2026 leaderboard as if they describe the same games.

In [ ]:
run_stage("report", max_seconds=120)
summary = json.loads((RUN / "summary.json").read_text())
print(json.dumps(summary, indent=2))
record = json.loads((KIT / "reports/latest_report.json").read_text())

## 6 · Interactive evidence

Eight charts are always produced. A passed gate adds two drop-one charts. Small calibration bins, correlated inputs and shared teams limit interpretation. The season-sensitivity table is descriptive; four already-used seasons with overlapping training sets do not warrant a reliable population confidence interval.

In [ ]:
plots = figures(RUN, KIT / "evidence")
print("Interactive figures:", len(plots))
for fig in plots[:4]:
    fig.show()

In [ ]:
for fig in plots[4:]:
    fig.show()
display(pd.read_csv(RUN / "season_sensitivity.csv").round(7))

## 7 · Save and stop this milestone

Press **Ctrl+S**. Download **`reports/milestone_05_return.zip`** below and return it to ChatGPT. Do not publish model files, alter thresholds, run the old shooting panel or generate a submission. The current submitted score remains **0.1222672** until a real new submission is evaluated; **0.1097454** remains a research target, not a promised outcome.

In [ ]:
display(FileLink(str(Path(record["return_zip"]).relative_to(KIT))))
display(FileLink(str(Path(record["html"]).relative_to(KIT))))
print("Return:", record["return_zip"])
print("Keep private_runs intact. Completed fits and feature snapshots are content-verified on resumption.")